**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/12-charuco-webcam-pipeline.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/watch-and-learn) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6L · ChArUco Calibration Pipeline (Nicolai Nielsen)

_Lecture 6, video lab · Nicolai Nielsen._

> 🎬 **Watch first:** [How to do Camera Calibration with ChArUco Boards](https://www.youtube.com/watch?v=EUvco3rjUdQ) (Nicolai Nielsen).
> Step-by-step notes for this video: [Tutorial 08 · ChArUco boards](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#charuco-video).

<a href="https://www.youtube.com/watch?v=EUvco3rjUdQ"><img src="https://img.youtube.com/vi/EUvco3rjUdQ/mqdefault.jpg" width="320" alt="How to do Camera Calibration with ChArUco Boards"></a>

## What the video does, in plain words

Capture 10–20 pictures of a **ChArUco** board with a webcam (**s** saves, **ESC** quits), feed them to a calibration tool,
and get a **JSON** with the camera matrix, distortion, and **one reprojection error per picture**. Then **remove the bad
pictures** and recalibrate, look at the **3-D view** of where each board was, and **undistort** (crop to the valid region).

The video uses his commercial web app. Everything it does is available **free in OpenCV**, and that is what we build here,
using the board from the video: **18 × 11 squares, `DICT_4X4_100`** (99 markers, each with its own ID).
So that it runs anywhere we render the pictures with a virtual camera (as in 6E) and spoil two of them on purpose:
the board **moved** while the rolling shutter read the rows, so it looks slightly sheared. Those are the outliers to remove.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

In [ ]:
import glob, json, os, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)
print("OpenCV", cv2.__version__)

In [ ]:
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_100)
SQUARE, MARKER = 30.0, 22.0                                    # mm, measure your print!
board = cv2.aruco.CharucoBoard((18, 11), SQUARE, MARKER, aruco_dict)
tex = board.generateImage((18 * 150, 11 * 150), marginSize=0)  # 0.2 mm per pixel
cv2.imwrite("charuco_18x11.png", board.generateImage((3600, 2200), marginSize=80))   # print this (A3 or larger)

K_true = np.array([[960.0, 0, 646.0], [0, 958.0, 364.0], [0, 0, 1]])
D_true = np.array([-0.09, 0.11, 0.0004, -0.0006, -0.05])
views = []
for k, (r, t) in enumerate(C.random_poses(16, seed=21, dist_mm=(560, 900), tilt_deg=30, board_mm=(540, 330))):
    v = C.render_board(tex, 0.2, K_true, D_true, r, t, (1280, 720), background=150)
    if k in (4, 11):   # two shaky shots: the board moved while the ROLLING shutter read the rows -> sheared + blurred
        v = cv2.warpAffine(v, np.float32([[1, 0.03, -11], [0, 1, 0]]), (1280, 720), borderValue=150)
    views.append(v)
fig, ax = plt.subplots(2, 8, figsize=(26, 4.5))
for a, v in zip(ax.ravel(), views): a.imshow(v, cmap="gray"); a.axis("off")
plt.show()

## 1. Capturing your own pictures (the video's `generate_images.py`)

Run this on your laptop (not Colab), with the printed board:

```python
cap = cv2.VideoCapture(0); num = 0
os.makedirs("output_images", exist_ok=True)
while True:
    ok, frame = cap.read()
    cv2.imshow("frame (s = save, ESC = quit)", frame)
    k = cv2.waitKey(1) & 0xFF
    if k == 27: break
    if k == ord("s"): cv2.imwrite(f"output_images/img{num}.png", frame); num += 1
cap.release(); cv2.destroyAllWindows()
```

## 2. Detect every marker and corner (with IDs)

In [ ]:
detector = cv2.aruco.CharucoDetector(board)
all_obj, all_img, used = [], [], []
for k, v in enumerate(views):
    ch_corners, ch_ids, m_corners, m_ids = detector.detectBoard(v)
    if ch_ids is not None and len(ch_ids) >= 10:
        obj, img = board.matchImagePoints(ch_corners, ch_ids)
        all_obj.append(obj); all_img.append(img); used.append(k)
print(f"{len(used)} views usable; corners per view: {[len(o) for o in all_obj]}")

vis = cv2.cvtColor(views[0], cv2.COLOR_GRAY2BGR)
ch_corners, ch_ids, m_corners, m_ids = detector.detectBoard(views[0])
cv2.aruco.drawDetectedMarkers(vis, m_corners, m_ids)
plt.figure(figsize=(14, 8)); plt.imshow(vis[:, :, ::-1]); plt.title("each marker has its own ID"); plt.axis("off"); plt.show()

## 3. Calibrate and look at the error of each picture

In [ ]:
size = (1280, 720)
rms, K, D, rvecs, tvecs, _, _, per_view = cv2.calibrateCameraExtended(all_obj, all_img, size, None, None)
per_view = per_view.ravel()
plt.figure(figsize=(12, 3.5)); plt.bar([str(u) for u in used], per_view, color=["#ea580c" if e > 2 * np.median(per_view) else "#28568a" for e in per_view])
plt.xlabel("image index"); plt.ylabel("RMS (px)"); plt.title(f"reprojection error per image (overall {rms:.3f} px)"); plt.show()

## 4. Remove the outliers and recalibrate (the video's main advice)

Drop every view whose error is **more than 2 × the median**, then calibrate again.

In [ ]:
keep = list(range(len(all_obj)))        # TODO: keep only views with per_view[i] <= 2 * median
rms2, K2, D2, rvecs2, tvecs2 = cv2.calibrateCamera([all_obj[i] for i in keep], [all_img[i] for i in keep], size, None, None)
print(f"removed {len(all_obj) - len(keep)} views: RMS {rms:.3f} -> {rms2:.3f} px")
print("fx error before %.2f %%, after %.2f %%" % (100 * abs(K[0, 0] - 960) / 960, 100 * abs(K2[0, 0] - 960) / 960))

In [ ]:
C.check_outlier_removal(rms, rms2, len(all_obj) - len(keep))

## 5. Export a JSON like the app and look at the boards in 3-D

In [ ]:
out = {"camera_matrix": K2.tolist(), "dist_coeffs": D2.ravel().tolist(), "rms": rms2, "image_size": size,
       "board": {"squares": [18, 11], "square_mm": SQUARE, "marker_mm": MARKER, "dictionary": "DICT_4X4_100"},
       "views": [{"index": int(used[i]), "rvec": rvecs2[n].ravel().tolist(), "tvec": tvecs2[n].ravel().tolist()} for n, i in enumerate(keep)]}
json.dump(out, open("charuco_calibration.json", "w"), indent=1)

fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
corners = np.array([[0, 0, 0], [540, 0, 0], [540, 330, 0], [0, 330, 0], [0, 0, 0]], float)
for r, t in zip(rvecs2, tvecs2):
    Rm, _ = cv2.Rodrigues(r); P = (Rm @ corners.T + t).T
    ax.plot(P[:, 0], P[:, 2], -P[:, 1])
ax.scatter([0], [0], [0], c="red", s=60); ax.set_xlabel("x (mm)"); ax.set_ylabel("z (mm)"); ax.set_zlabel("-y")
ax.set_title("where each board was (camera = red dot)"); plt.show()

## 6. Undistort and crop to the valid region

In [ ]:
newK, roi = cv2.getOptimalNewCameraMatrix(K2, D2, size, 1, size)
und = cv2.undistort(views[0], K2, D2, None, newK)
x, y, rw, rh = roi
fig, ax = plt.subplots(1, 2, figsize=(18, 5)); ax[0].imshow(views[0], cmap="gray"); ax[0].set_title("original")
ax[1].imshow(und[y:y + rh, x:x + rw], cmap="gray"); ax[1].set_title("undistorted, cropped to roi")
for a in ax: a.axis("off")
plt.show()

## Your turn

1. Print `charuco_18x11.png` (or show it full-screen on a tablet, as Nicolai does with the chessboard), capture 15 pictures
   with section 1, and run sections 2–6 on them (`views = [cv2.imread(f, 0) for f in sorted(glob.glob("output_images/*.png"))]`).
2. The video says a few millimetres of board error "make a huge difference". Re-run section 3 with `SQUARE = 31.0` (a
   3 % wrong measurement) for the object points. Which numbers change: K, D or the tvecs?
3. Why is a ChArUco board better than a chessboard for covering the **corners** of a wide-angle image?